# FOI Sentinel — The Engine Room

A back-end walkthrough of how **FOI Sentinel** processes a UK local-authority information request end-to-end on **Snowflake Cortex**. Every cell runs live against `FOI.FOI_SENTINEL_V2`.

**Pipeline shown:** intake & triage → prioritisation → retrieval (RAG) → grounded drafting → SAR redaction → cost & audit → **model bake-off** (which Cortex model per task).

> Data is synthetic/demo where personal; peer disclosures (Camden, GLA, WhatDoTheyKnow) are real published data.

> **Residency:** this account is in AWS `us-west-2`. For an EU-inference posture, cross-region routing can be pinned to the EU (`AWS_EU`) — see the Residency cell below.

In [ ]:
# Session context (DDL/admin — the sanctioned session.sql() use)
from snowflake.snowpark.context import get_active_session
session = get_active_session()
for stmt in ["USE WAREHOUSE FOI_WH", "USE DATABASE FOI", "USE SCHEMA FOI_SENTINEL_V2"]:
    session.sql(stmt).collect()
print("Context:", session.sql("SELECT CURRENT_ACCOUNT(), CURRENT_REGION(), CURRENT_SCHEMA()").collect()[0])

## Residency — where inference runs

`CORTEX_ENABLED_CROSS_REGION` controls where the *inference payload* is processed (stored data always stays in the account's home region). To keep inference in the EU for a UK client, an `ACCOUNTADMIN` sets:

```sql
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_EU';
```

Under `AWS_EU`: native EU models (mistral, llama, mixtral) + Claude via EU cross-region are available; OpenAI GPT and snowflake-llama are US-only and drop out. The cell below shows the current setting.

In [ ]:
%%sql -r cross_region
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT

## 1. Intake & triage

A raw request runs through two Cortex calls in one round-trip: `SENTIMENT` (requester tone) and `COMPLETE` (structured triage — regime, priority, complexity, factors). This is what the Intake page does per inbound email.

In [ ]:
%%sql -r triage
SELECT
  SNOWFLAKE.CORTEX.SENTIMENT(
    'Please provide the total the council spent on temporary accommodation for homeless households in each of the last three financial years, broken down by financial year, and the number of households in temporary accommodation at each year end.'
  ) AS requester_sentiment,
  SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
    'You are a UK local-government FOI officer. Return STRICT JSON only with keys category (FOI/EIR/SAR/BAU), priority (HIGH/MEDIUM/LOW), complexity_score (number 1-10), complexity_factors (array of short phrases), suggested_departments (array), estimated_hours (number), summary (one sentence). REQUEST: Please provide the total the council spent on temporary accommodation for homeless households in each of the last three financial years, broken down by financial year, and the number of households in temporary accommodation at each year end. JSON only.'
  ) AS triage_json

## 2. Prioritisation — deterministic clock + AI signals

The Red/Amber/Green is **not** an AI guess — it is derived from the statutory deadline against a working-day calendar. Complexity and sentiment (from triage) are decision *aids* layered on top. Here are live open cases ordered by how close they are to the 20-working-day deadline.

In [ ]:
%%sql -r priority_cases
SELECT
  REFERENCE,
  REGIME,
  CURRENT_STAGE,
  STATUTORY_DEADLINE,
  DATEDIFF('day', CURRENT_DATE(), STATUTORY_DEADLINE) AS days_to_deadline,
  COMPLEXITY_RANK,
  SENTIMENT_SCORE
FROM FOI_CASE
WHERE STATUS = 'OPEN'
ORDER BY STATUTORY_DEADLINE
LIMIT 10

## 3. Retrieval — Cortex Search (the RAG layer)

Grounding comes from managed **Cortex Search** services over peer disclosures and the council's own records. Below: the closest WhatDoTheyKnow precedents to our request, retrieved semantically and flattened into rows.

In [ ]:
%%sql -r retrieval
SELECT
  f.value:AUTHORITY_NAME::string AS authority,
  f.value:OUTCOME::string        AS outcome,
  f.value:REQUEST_TITLE::string  AS title,
  LEFT(f.value:SNIPPET::string, 180) AS snippet
FROM TABLE(FLATTEN(input => PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'FOI.FOI_SENTINEL_V2.WDTK_PRECEDENT_SEARCH',
    '{"query":"council spend on temporary accommodation for homeless households by year","columns":["AUTHORITY_NAME","OUTCOME","REQUEST_TITLE","SNIPPET"],"limit":4}'
  )
):results)) f

## 4. Grounded drafting

The council's **own** internal-holdings facts are retrieved, assembled into a cited source block, and passed to `COMPLETE` with instructions to use only those sources and cite them `[S1]`, `[S2]`. This is the real `suggestAnswer` pattern — grounded, not free-generated.

In [ ]:
# Retrieve the council's own facts for the matched theme, build a cited source block, then draft.
import json

spec = json.dumps({"query": "temporary accommodation spend by financial year", "columns": ["FACT_TEXT", "THEME", "PERIOD"], "limit": 6})
spec_sql = spec.replace("'", "''")
res = session.sql(f"SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW('FOI.FOI_SENTINEL_V2.INTERNAL_HOLDINGS_SEARCH', '{spec_sql}') AS R").collect()[0]["R"]
facts = json.loads(res).get("results", [])

sources = "\n".join(f"[S{i+1}] (This council's records) {r.get('FACT_TEXT','')}" for i, r in enumerate(facts))
print("SOURCES USED:\n", sources or "(none)", "\n")

prompt = (
    "You are an FOI officer at Exampleton Council drafting a SUGGESTED answer for review. "
    "Use ONLY the sources below and cite them inline as [S1], [S2]. Disclose specific figures and periods directly. "
    "Do not invent figures. 120-180 words.\n\nREQUEST: total spent on temporary accommodation for homeless households "
    "in each of the last three financial years.\n\nSOURCES:\n" + (sources or "(no matches)") + "\n\nSuggested answer:"
)
prompt_sql = prompt.replace("'", "''")
draft = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', '{prompt_sql}') AS R").collect()[0]["R"]
print("GROUNDED DRAFT:\n", draft)

## 5. SAR redaction (the flagship)

For a Subject Access Request the requester gets **their own** data, but third-party personal data must be removed (s.40 / DPA 2018). `AI_PARSE_DOCUMENT` reads the staged case-file PDF; `AI_EXTRACT` (with `scores => TRUE`) *selectively* detects third parties while keeping the requester (James Whitfield). Note we do **not** use `AI_REDACT` — it would mask the requester's own data too.

In [ ]:
%%sql -r sar
SELECT
  LEFT(TO_VARCHAR(AI_PARSE_DOCUMENT(
    TO_FILE('@FOI.FOI_SENTINEL_V2.SAR_STAGE', 'sar_casefile.pdf'), {'mode': 'LAYOUT'}):content), 400) AS parsed_preview,
  TO_JSON(AI_EXTRACT(
    file => TO_FILE('@FOI.FOI_SENTINEL_V2.SAR_STAGE', 'sar_casefile.pdf'),
    responseFormat => { 'schema': { 'type':'object', 'properties': {
      'third_party_names':  {'type':'array','description':'Full names of people who are NOT the claimant James Whitfield'},
      'third_party_phones': {'type':'array','description':'Personal or direct telephone numbers that are not the claimant own number; exclude the council published switchboard'}
    }}},
    scores => TRUE
  )) AS third_party_findings

## 6. Cost & audit trail

**Cost:** `COUNT_TOKENS` measures the tokens each stage consumes — the basis for a real per-request £ figure. **Audit:** every AI vs human decision is written to `FOI_CASE_EVENT`, so any output is traceable at an ICO review.

In [ ]:
%%sql -r token_cost
SELECT
  'triage prompt' AS stage,
  SNOWFLAKE.CORTEX.COUNT_TOKENS('mistral-large2',
    'You are a UK local-government FOI officer. Return STRICT JSON for the temporary accommodation spend request over the last three financial years.'
  ) AS prompt_tokens

In [ ]:
%%sql -r audit_trail
SELECT EVENT_TS, ACTOR_TYPE, ACTOR, EVENT_TYPE, NOTE
FROM FOI_CASE_EVENT
ORDER BY EVENT_TS DESC
LIMIT 12

## 7. Model bake-off — which Cortex model per task?

The same **triage classification** task is run across candidate models. We measure **latency**, **prompt tokens**, and eyeball the **output**. Native EU models run first; Claude (EU cross-region) is attempted and degrades gracefully if not enabled. Pick the cheapest/fastest model that gets the task right — you rarely need a frontier model to classify a request.

In [ ]:
import time, pandas as pd

MODELS = [
    'mistral-7b', 'mixtral-8x7b', 'llama3.1-8b', 'llama3.1-70b', 'mistral-large2',
    'claude-haiku-4-5', 'claude-sonnet-4-6',
]
task_prompt = (
    'You are a UK local-government FOI triage assistant. Classify this request. '
    'Return ONLY strict JSON with keys category (FOI/EIR/SAR/BAU), priority (HIGH/MEDIUM/LOW), complexity_score (1-10). '
    'REQUEST: Please provide the total the council spent on temporary accommodation for homeless households in each of '
    'the last three financial years, broken down by financial year. JSON only.'
)
p = task_prompt.replace("'", "''")
rows = []
for m in MODELS:
    try:
        t0 = time.time()
        r = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{m}', '{p}') AS R").collect()[0]["R"]
        dt = round(time.time() - t0, 2)
        try:
            toks = session.sql(f"SELECT SNOWFLAKE.CORTEX.COUNT_TOKENS('{m}', '{p}') AS T").collect()[0]["T"]
        except Exception:
            toks = None
        rows.append({"model": m, "latency_s": dt, "prompt_tokens": toks, "output": " ".join(str(r).split())[:140]})
    except Exception as e:
        rows.append({"model": m, "latency_s": None, "prompt_tokens": None, "output": f"unavailable: {str(e)[:90]}"})

bakeoff = pd.DataFrame(rows)
bakeoff

In [ ]:
import matplotlib.pyplot as plt

avail = bakeoff.dropna(subset=["latency_s"]).sort_values("latency_s")
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(avail["model"], avail["latency_s"], color="#1d70b8")
ax.set_xlabel("Latency (s) — classification task")
ax.set_title("Model latency for FOI triage classification (lower is better)")
for i, v in enumerate(avail["latency_s"]):
    ax.text(v, i, f" {v}s", va="center")
plt.tight_layout()
plt.show()

unavailable = bakeoff[bakeoff["latency_s"].isna()]["model"].tolist()
if unavailable:
    print("Unavailable in this region/config (expected if EU cross-region not enabled):", unavailable)

## Recommendation & takeaways

- **Classification / priority / complexity:** a small native model (`mistral-7b`, `llama3.1-8b`) or the purpose-built `AI_CLASSIFY` is usually enough — far cheaper and faster than `mistral-large2`. Reserve large models for drafting.
- **Grounded drafting:** `mistral-large2` (native EU) is a strong default; `claude-sonnet-4-6` (EU cross-region) is the frontier comparator when quality matters most.
- **Extraction / redaction:** `AI_EXTRACT` (arctic-extract) is fixed-purpose and native — no model choice needed.
- **Residency:** everything above runs under `AWS_EU` cross-region except OpenAI GPT / snowflake-llama (US-only).
- **Honesty:** demo/synthetic personal data; this account's home region is `us-west-2`, so `AWS_EU` routes inference to the EU while stored data stays in the US — full residency needs an EU-home account.